In [ ]:
import sys
import os

print("====== STEP 1: CLONING BENCHMARK AND AA-CLIP ======")
# 1. Define public repository details
BENCHMARK_REPOSITORY = "Parsagh05/Natural-Corruption-Robustness"
repo_url = f"https://github.com/{BENCHMARK_REPOSITORY}.git"
target_dir = "/kaggle/working/Natural-Corruption-Robustness"
aaclip_dir = "/kaggle/working/AA-CLIP"

# 3. Clone/update this benchmark repository
if not os.path.exists(target_dir):
    print(f"Cloning public benchmark repository into {target_dir}...")
    clone_status = os.system(f"git clone {repo_url} {target_dir}")
    if clone_status == 0:
        print("Repository cloned successfully.")
    else:
        raise RuntimeError("Git clone failed. Check your token permissions.")
else:
    print("Repository directory already exists. Pulling latest changes...")
    pull_status = os.system(f"git -C {target_dir} pull --ff-only")
    if pull_status == 0:
        print("Repository updated successfully.")
    else:
        print("Repository update skipped or failed. Continuing with existing checkout.")

# 4. Clone/update the official AA-CLIP implementation used by the wrapper
if not os.path.exists(aaclip_dir):
    print(f"Cloning official AA-CLIP into {aaclip_dir}...")
    clone_status = os.system(f"git clone https://github.com/Mwxinnn/AA-CLIP.git {aaclip_dir}")
    if clone_status == 0:
        print("AA-CLIP cloned successfully.")
    else:
        raise RuntimeError("Failed to clone https://github.com/Mwxinnn/AA-CLIP")
else:
    print("AA-CLIP directory already exists. Pulling latest changes...")
    pull_status = os.system(f"git -C {aaclip_dir} pull --ff-only")
    if pull_status == 0:
        print("AA-CLIP updated successfully.")
    else:
        print("AA-CLIP update skipped or failed. Continuing with existing checkout.")

# 5. Register path environment for Python
if aaclip_dir not in sys.path:
    sys.path.insert(0, aaclip_dir)
    print("Registered AA-CLIP path to sys.path.")

print("\n====== STEP 2: INSTALLING SYSTEM DEPENDENCIES ======")
os.system("pip install -q open-clip-torch scipy opencv-python scikit-learn scikit-image ftfy regex tqdm tabulate timm==0.6.12 torchsummary seaborn dash-table thop kornia==0.6.9 ipdb tiktoken einops")
print("Core dependencies installed successfully.")
print("\nENVIRONMENT READY.")


```python 
UNCATEGORIZED_CORRUPTION_TYPES = [
    # "gaussian_noise", # visa 20 min
    # "shot_noise", # visa 30 min
    # "impulse_noise", # visa 20 min
    # "defocus_blur", # visa 25 min
    # "motion_blur", # visa 1.20 hrs
    # "zoom_blur", # visa 2.20 hrs
    # "brightness", # visa 45 min
    # "contrast", # visa 20 min
]
CATEGORIZED_CORRUPTION_TYPES = [
    # "noise", # visa 25 min
    # "blur", # visa 1.30 min
    # "photometric", # 30 min
    # "geometric" # 25 min
]
```

In [ ]:
import sys
import os
import gc
import hashlib
import urllib.request
from pathlib import Path
import torch

# 1. Set the required environment variable for AA-CLIP.
os.environ["AACLIP_ROOT"] = "/kaggle/working/AA-CLIP"

# 2. Register the repository root for shared assets and the zero-shot harness.
benchmark_path = Path('/kaggle/working/Natural-Corruption-Robustness')
harness_path = benchmark_path / 'zero_shot'
for import_path in (benchmark_path, harness_path):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

# Ensure the cloned AA-CLIP path is also visible to Python.
aaclip_path = '/kaggle/working/AA-CLIP'
if aaclip_path not in sys.path:
    sys.path.insert(0, aaclip_path)

from shared import corruption_plan_path
from harness.config import CLEAN_CONDITION
from harness.runner import run_evaluation

# ==============================================================================
# DATASET & MODEL CONFIGURATION
# ==============================================================================
# Choose exactly one dataset for this notebook session: "mvtec" or "visa".
# DATASET_NAME = "mvtec"
DATASET_NAME = "visa"

MODEL_NAME = "AA-CLIP"
# False: evaluate every concrete corruption independently. True: evaluate the
# balanced categories below, assigning each image one operation in that group.
USE_CATEGORIZED_CORRUPTIONS = False
# Must match base_seed in demo_categorized.ipynb and its CSV generator.
CATEGORIZED_CORRUPTION_SEED = 123

UNCATEGORIZED_CORRUPTION_TYPES = [
    "gaussian_noise",
    "shot_noise",
    "impulse_noise",
    "defocus_blur",
    "motion_blur",
    "zoom_blur",
    "brightness",
    "contrast",
]

CATEGORIZED_CORRUPTION_TYPES = [
    "noise",
    "blur",
    "photometric",
    "geometric"
]

CORRUPTION_TYPES = (
    CATEGORIZED_CORRUPTION_TYPES
    if USE_CATEGORIZED_CORRUPTIONS else UNCATEGORIZED_CORRUPTION_TYPES
)
# Set this to False to skip evaluation on original, uncorrupted images. Keep
# severity 0 out of SEVERITY_LEVELS; it is reserved for this clean baseline.
INCLUDE_CLEAN_BASELINE = True
ZERO_CORRUPTION_CONDITION = CLEAN_CONDITION
SEVERITY_LEVELS = [1, 2, 3, 4]
BATCH_SIZE = 2  # Increase to 8 or 16 on larger Kaggle GPUs if memory allows.
CORRUPTION_CACHE_FORMAT = "png"  # Use "jpeg" if you want ImageNet-C-style cached files.

# ==============================================================================
# ENVIRONMENT PATH CONFIGURATION
# ==============================================================================
MVTEC_PATH = "/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection"
VISA_PATH = "/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922"
OUTPUT_ROOT = "/kaggle/working/outputs"
# CORRUPTION_CACHE_ROOT = "/kaggle/working/corruption_cache"
CORRUPTION_CACHE_ROOT = None
AACLIP_ROOT = "/kaggle/working/AA-CLIP"

DATASET_NAME = DATASET_NAME.lower().strip()
if DATASET_NAME not in {"mvtec", "visa"}:
    raise ValueError("DATASET_NAME must be either 'mvtec' or 'visa'.")

IS_MVTEC = DATASET_NAME == "mvtec"
selected_dataset = "MVTec AD" if IS_MVTEC else "VisA"
CORRUPTION_PLAN = str(corruption_plan_path(DATASET_NAME))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OPENCLIP_WEIGHT_URL = "https://openaipublic.azureedge.net/clip/models/3035c92b350959924f9f00213499208652fc7ea050643e8b385c2dac08641f02/ViT-L-14-336px.pt"
OPENCLIP_WEIGHT_SHA256 = "3035c92b350959924f9f00213499208652fc7ea050643e8b385c2dac08641f02"

# AA-CLIP needs two local assets: the trained adapter save directory
# containing image_adapter.pth/text_adapter.pth, and OpenAI's
# ViT-L-14-336px.pt weight.
def _valid_path(value):
    if not value:
        return None
    path = Path(value)
    return path if str(path) != "." and path.exists() else None


def _kaggle_input_roots():
    input_root = Path("/kaggle/input")
    return sorted(input_root.iterdir()) if input_root.exists() else []


def _sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as file_obj:
        for chunk in iter(lambda: file_obj.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _download_openclip_weight(destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temp_path = destination.with_suffix(destination.suffix + ".download")
    if temp_path.exists():
        temp_path.unlink()

    print(f"Downloading OpenCLIP ViT-L-14-336px weight to {destination}...")
    urllib.request.urlretrieve(OPENCLIP_WEIGHT_URL, temp_path)
    actual_sha256 = _sha256_file(temp_path)
    if actual_sha256 != OPENCLIP_WEIGHT_SHA256:
        temp_path.unlink(missing_ok=True)
        raise RuntimeError(
            "Downloaded ViT-L-14-336px.pt failed SHA-256 validation. "
            f"Expected {OPENCLIP_WEIGHT_SHA256}, got {actual_sha256}."
        )
    temp_path.replace(destination)
    print("OpenCLIP weight downloaded and verified.")
    return destination


AACLIP_CHECKPOINT_ROOT = Path(
    "/kaggle/input/datasets/parsagh1383/aa-clip-checkpoints-main/AA-CLIP_Checkpoints"
)
# AA-CLIP zero-shot evaluation must use adapters trained on the other dataset.
AACLIP_TRAIN_DATASET = "VisA" if IS_MVTEC else "MVTec"
if AACLIP_TRAIN_DATASET.lower() == DATASET_NAME:
    raise RuntimeError(
        "AA-CLIP checkpoint leakage: zero-shot evaluation requires "
        "a checkpoint trained on a different dataset."
    )
AACLIP_TRAIN_DIR = f"TrainOn{AACLIP_TRAIN_DATASET}"
AACLIP_CHECKPOINT_DIR = AACLIP_CHECKPOINT_ROOT / AACLIP_TRAIN_DIR
AACLIP_IMAGE_ADAPTER = AACLIP_CHECKPOINT_DIR / "image_adapter.pth"
AACLIP_TEXT_ADAPTER = AACLIP_CHECKPOINT_DIR / "text_adapter.pth"

if not AACLIP_IMAGE_ADAPTER.exists():
    raise FileNotFoundError(
        "AA-CLIP image adapter not found. Expected the Kaggle input at:\n"
        f"  {AACLIP_IMAGE_ADAPTER}"
    )
if not AACLIP_TEXT_ADAPTER.exists():
    print(f"WARNING: text_adapter.pth not found at {AACLIP_TEXT_ADAPTER}")

AACLIP_CHECKPOINT = str(AACLIP_CHECKPOINT_DIR)

clip_weight_candidates = [
    _valid_path(os.environ.get("AACLIP_CLIP_WEIGHT")),
    Path(AACLIP_ROOT) / "model/ViT-L-14-336px.pt",
]
for root in _kaggle_input_roots():
    clip_weight_candidates.extend(root.rglob("ViT-L-14-336px.pt"))
clip_weight_candidates = [path for path in clip_weight_candidates if path is not None]
AACLIP_CLIP_WEIGHT = next(
    (str(path) for path in clip_weight_candidates if path.exists()),
    None,
)
if AACLIP_CLIP_WEIGHT is None:
    try:
        AACLIP_CLIP_WEIGHT = str(
            _download_openclip_weight(Path(AACLIP_ROOT) / "model/ViT-L-14-336px.pt")
        )
    except Exception as exc:
        expected = "\n".join(f"  - {path}" for path in clip_weight_candidates if str(path))
        raise FileNotFoundError(
            "AA-CLIP CLIP backbone weight ViT-L-14-336px.pt not found and "
            "automatic download failed. Enable Kaggle internet or add the file "
            "as a Kaggle input. Checked:\n"
            f"{expected}\n"
            f"Download URL: {OPENCLIP_WEIGHT_URL}\n"
            f"Original download error: {exc!r}"
        ) from exc

model_kwargs = {
    "AA-CLIP": {
        "aaclip_root": AACLIP_ROOT,
        "checkpoint_path": AACLIP_CHECKPOINT,
        "clip_weight_path": AACLIP_CLIP_WEIGHT,
        "image_size": 518,
        "text_adapt_weight": 0.1,
        "image_adapt_weight": 0.1,
        "text_adapt_until": 3,
        "image_adapt_until": 6,
        "levels": [6, 12, 18, 24],
        "relu": False,
    }
}

print("LAUNCHING ISOLATED BENCHMARK")
print(f"Model:      {MODEL_NAME}")
print(f"Dataset:    {selected_dataset}")
print(f"Zero corruption: {f'{ZERO_CORRUPTION_CONDITION[0]} @ severity {ZERO_CORRUPTION_CONDITION[1]}' if INCLUDE_CLEAN_BASELINE else 'disabled'}")
print(f"Corruption: {CORRUPTION_TYPES} @ severities {SEVERITY_LEVELS}")
print(f"Categorized protocol: {USE_CATEGORIZED_CORRUPTIONS}")
if USE_CATEGORIZED_CORRUPTIONS:
    print(f"Corruption plan: {CORRUPTION_PLAN}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Device:     {DEVICE}")
print(f"Adapter:    {AACLIP_CHECKPOINT}")
print(f"Trained on: {AACLIP_TRAIN_DATASET} (cross-dataset zero-shot)")
print(f"CLIP:       {AACLIP_CLIP_WEIGHT}")
print(f"Outputs:    {OUTPUT_ROOT}")
print(f"Cache:      {CORRUPTION_CACHE_ROOT} ({CORRUPTION_CACHE_FORMAT})\n")

# ==============================================================================
# CONDITIONAL EVALUATION EXECUTION
# ==============================================================================
if IS_MVTEC:
    print(f"--- Starting isolated MVTec AD run for {MODEL_NAME} ---")
    try:
        run_evaluation(
            mvtec_root=MVTEC_PATH,
            visa_root=None,
            output_root=OUTPUT_ROOT,
            models=[MODEL_NAME],
            model_kwargs=model_kwargs,
            device=DEVICE,
            dataset="mvtec",
            corruption_types=CORRUPTION_TYPES,
            severity_levels=SEVERITY_LEVELS,
            include_clean=INCLUDE_CLEAN_BASELINE,
            batch_size=BATCH_SIZE,
            corruption_cache_root=CORRUPTION_CACHE_ROOT,
            corruption_cache_format=CORRUPTION_CACHE_FORMAT,
            categorized_corruptions=USE_CATEGORIZED_CORRUPTIONS,
            categorized_corruption_plans={DATASET_NAME: CORRUPTION_PLAN},
            corruption_seed=(CATEGORIZED_CORRUPTION_SEED if USE_CATEGORIZED_CORRUPTIONS else None),
        )
        print(f"Successful MVTec evaluation completed for {MODEL_NAME}.")
    except Exception as e:
        print(f"ERROR during MVTec run for {MODEL_NAME}: {str(e)}")
        raise

else:
    print(f"--- Starting isolated VisA run for {MODEL_NAME} ---")
    try:
        run_evaluation(
            mvtec_root=None,
            visa_root=VISA_PATH,
            output_root=OUTPUT_ROOT,
            models=[MODEL_NAME],
            model_kwargs=model_kwargs,
            device=DEVICE,
            dataset="visa",
            corruption_types=CORRUPTION_TYPES,
            severity_levels=SEVERITY_LEVELS,
            include_clean=INCLUDE_CLEAN_BASELINE,
            batch_size=BATCH_SIZE,
            corruption_cache_root=CORRUPTION_CACHE_ROOT,
            corruption_cache_format=CORRUPTION_CACHE_FORMAT,
            categorized_corruptions=USE_CATEGORIZED_CORRUPTIONS,
            categorized_corruption_plans={DATASET_NAME: CORRUPTION_PLAN},
            corruption_seed=(CATEGORIZED_CORRUPTION_SEED if USE_CATEGORIZED_CORRUPTIONS else None),
        )
        print(f"Successful VisA evaluation completed for {MODEL_NAME}.")
    except Exception as e:
        print(f"ERROR during VisA run for {MODEL_NAME}: {str(e)}")
        raise

# ==============================================================================
# POST-RUN CLEANUP
# ==============================================================================
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n" + "="*70)
print(f"EXECUTION FINISHED FOR {MODEL_NAME} ON {selected_dataset}")
print(f"Check your '{OUTPUT_ROOT}' directory to collect the generated CSV files.")
print("="*70)
